In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import scanpy as sc

In [ ]:
import xgboost as xgb
print("xgboost version:", xgb.__version__)

xgboost version: 2.1.4


In [5]:
# Load file
FILE_PATH = "../data/Variant_Vax_obj.h5ad"
adata = sc.read_h5ad(FILE_PATH)
adata

AnnData object with n_obs × n_vars = 48730 × 29961
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'Variant_Group', 'Participant', 'SARSCoV2_PCR_Status', 'Vaccination_Status', 'WHO_Score_at_Peak', 'SingleCell_SARSCoV2_RNA_Status', 'Coarse_Annotation', 'Detailed_Annotation', 'Variant_Vax_Group'
    obsm: 'X_harmony', 'X_pca', 'X_umap'

In [8]:
import numpy as np
import pandas as pd


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

# -----------------------------
# 1. Load AnnData and build patient-level labels
# -----------------------------

obs = adata.obs[["Participant", "WHO_Score_at_Peak"]].copy()

# Check that WHO is constant per participant (sanity check)
per_patient = (
    obs.groupby("Participant")["WHO_Score_at_Peak"]
    .agg(["nunique", "min", "max"])
)
if (per_patient["nunique"] > 1).any():
    raise ValueError("Some participants have multiple WHO_Score_at_Peak values.")

# Collapse to one row per participant
patient_labels = (
    obs.groupby("Participant")["WHO_Score_at_Peak"]
    .first()
    .reset_index()
)

# 3-class encoding: 0=no covid, 1=mild (1-4), 2=severe (5-8)
def encode_severity(who):
    if who == 0:
        return 0
    elif 1 <= who <= 4:
        return 1
    elif 5 <= who <= 8:
        return 2
    else:
        return np.nan

patient_labels["severity_class"] = patient_labels["WHO_Score_at_Peak"].apply(encode_severity)
patient_labels = patient_labels.dropna(subset=["severity_class"])
patient_labels["severity_class"] = patient_labels["severity_class"].astype(int)

print("Patient label table shape:", patient_labels.shape)
print(patient_labels["severity_class"].value_counts().sort_index())

# -----------------------------
# 2. Select 10 random participants (Option C)
# -----------------------------
rng = np.random.default_rng(42)
unique_participants = patient_labels["Participant"].unique()
if len(unique_participants) < 10:
    raise ValueError("Fewer than 10 participants available.")

selected_participants = rng.choice(unique_participants, size=10, replace=False)
patient_labels_10 = patient_labels[patient_labels["Participant"].isin(selected_participants)].reset_index(drop=True)

print("\nSelected 10 participants:")
print(patient_labels_10[["Participant", "WHO_Score_at_Peak", "severity_class"]])

# -----------------------------
# 3. Load dummy latent states and attach to these 10 patients
# -----------------------------
# Assumes dummy_lat_state.csv is in the same folder as the notebook
latents = pd.read_csv("dummy_lat_state.csv")

if latents.shape[0] != 10:
    raise ValueError(f"Expected 10 rows in dummy_lat_state.csv, found {latents.shape[0]}.")

expected_cols = [f"z{i}" for i in range(1, 21)]
if list(latents.columns) != expected_cols:
    raise ValueError(f"Expected columns {expected_cols}, found {list(latents.columns)}.")

# Attach latents to the 10 selected patients (row-wise)
patient_features = patient_labels_10.copy()
for i, col in enumerate(expected_cols):
    patient_features[col] = latents[col].values

print("\nPatient feature table (10 x 20) head:")
print(patient_features[["Participant", "severity_class"] + expected_cols[:5]].head())

# -----------------------------
# 4. Prepare X, y
# -----------------------------
feature_cols = expected_cols
X = patient_features[feature_cols].values
y = patient_features["severity_class"].values

print("\nX shape:", X.shape, "y shape:", y.shape)

# -----------------------------
# 5. Stratified CV with dynamic n_splits (to avoid class-count issues)
# -----------------------------
class_counts = pd.Series(y).value_counts()
min_class_count = class_counts.min()
n_splits = min(5, min_class_count)  # ensure each class appears in each fold

if n_splits < 2:
    raise ValueError(f"Not enough samples per class for CV. Class counts:\n{class_counts}")

print(f"\nUsing StratifiedKFold with n_splits={n_splits}")
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

fold_results = []
all_pred_rows = []

fold_idx = 0
for train_idx, val_idx in skf.split(X, y):
    fold_idx += 1
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    clf = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=20,
        verbose=False,
    )

    y_val_pred = clf.predict(X_val)
    y_val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, y_val_pred)
    f1_macro = f1_score(y_val, y_val_pred, average="macro")

    fold_results.append({"fold": fold_idx, "accuracy": acc, "f1_macro": f1_macro})

    # Store predictions with participant IDs
    val_participants = patient_features.iloc[val_idx]["Participant"].values
    for pid, true_label, pred_label, proba in zip(val_participants, y_val, y_val_pred, y_val_proba):
        all_pred_rows.append({
            "fold": fold_idx,
            "Participant": pid,
            "true_class": int(true_label),
            "pred_class": int(pred_label),
            "prob_class_0": float(proba[0]),
            "prob_class_1": float(proba[1]),
            "prob_class_2": float(proba[2]),
        })

    print(f"Fold {fold_idx}: accuracy={acc:.3f}, f1_macro={f1_macro:.3f}")

# -----------------------------
# 6. Summarize CV results
# -----------------------------
results_df = pd.DataFrame(fold_results)
print("\nCV results:")
print(results_df)
print("\nMean accuracy: {:.3f} ± {:.3f}".format(results_df["accuracy"].mean(), results_df["accuracy"].std()))
print("Mean f1_macro: {:.3f} ± {:.3f}".format(results_df["f1_macro"].mean(), results_df["f1_macro"].std()))

# -----------------------------
# 7. Train final model on all 10 patients and export feature importance
# -----------------------------
final_clf = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

final_clf.fit(X, y, verbose=False)

importances = final_clf.feature_importances_
fi_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances,
}).sort_values("importance", ascending=False)

print("\nTop feature importances:")
print(fi_df.head(10))

# -----------------------------
# 8. Save predictions and feature importance
# -----------------------------
pred_df = pd.DataFrame(all_pred_rows)
pred_df.to_csv("xgb_patient_predictions.csv", index=False)
fi_df.to_csv("xgb_feature_importance.csv", index=False)

print("\nSaved:")
print(" - xgb_patient_predictions.csv")
print(" - xgb_feature_importance.csv")

# -----------------------------
# 9. Confusion matrix for the last fold (just for a quick look)
# -----------------------------
last_fold = max(pred_df["fold"])
last_fold_df = pred_df[pred_df["fold"] == last_fold]
cm = confusion_matrix(last_fold_df["true_class"], last_fold_df["pred_class"], labels=[0,1,2])


Patient label table shape: (112, 3)
severity_class
0    27
1    31
2    54
Name: count, dtype: int64

Selected 10 participants:
    Participant  WHO_Score_at_Peak  severity_class
0  A_COVID19_10                  5               2
1  A_COVID19_11                  8               2
2  A_COVID19_23                  8               2
3    CONTROL_15                  0               0
4  D_COVID19_11                  7               2
5  D_COVID19_19                  8               2
6  D_COVID19_23                  8               2
7  O_COVID19_02                  1               1
8  O_COVID19_16                  5               2
9  O_COVID19_18                  8               2

Patient feature table (10 x 20) head:
    Participant  severity_class    z1    z2    z3    z4    z5
0  A_COVID19_10               2 -0.12  1.44  0.55 -2.11  0.03
1  A_COVID19_11               2  0.77 -1.22  2.11  0.44 -0.55
2  A_COVID19_23               2  1.55  0.33 -0.44  2.55 -1.88
3    CONTROL_15         

ValueError: Not enough samples per class for CV. Class counts:
2    8
0    1
1    1
Name: count, dtype: int64